# 5.6. Dropout

Deep neural networks tend to _overfit_ as their complexity grows. This means they may have memorized all the training data but unable to generalize effectively to unseen inputs. Dropout is a technique aimed to alleviate this issue by "forgetting" a subset of neurons in each hidden layer during training to avoid overfitting.

In [1]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 5.6.2. Implementation from Scratch

We won't implement dropout from scratch. However, we'll first observe the effects of dropout on individual, synthetic tensors absent any model training and inference to get a high-level idea of what it does.

Given a hidden layer with $n$ activations and a dropout probability of $p$, approximately $np$ activations will be zeroed while the rest are scaled by a factor of $\frac{1}{1 - p}$. This ensures the expected value of the activations remain unchanged.

For example, with $n = 100$ activations and a dropout probability of $p = 0.2$:

1. Approximately $100 \times 0.2 = 20$ activations will be zeroed
1. The remaining \(approx.\) $100 - 20 = 80$ activations are scaled by a factor of exactly $\frac{1}{1 - 0.2} = 1.25$

Usually, the dropout value is set closer to $0$ for hidden layers close to the input layer, e.g. $0.2$, and larger close to the output layer, e.g. $0.5$. Overly small values of dropout may be ineffective to prevent overfitting while overly large values of dropout may cause underfitting.

In [2]:
import mindspore.ops as ops
from mindspore import dtype as mstype

X = ops.arange(16, dtype=mstype.float32)
X

/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
2026-04-25 14:52:29.538730: E external/org_tensorflow/tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute is_closed which is not in the op definition: Op<name=Range; signature=start:Tidx, limit:Tidx, delta:Tidx -> output:Tidx; attr=Tidx:type,default=DT_INT32,allowed=[DT_BFLOAT16, DT_HALF, DT_FLOAT, DT_DOUBLE, DT_INT8, DT_INT16, DT_INT32, DT_INT64, DT_UINT16, DT_UINT32]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Range1}}


Tensor(shape=[16], dtype=Float32, value= [ 0.00000000e+00,  1.00000000e+00,  2.00000000e+00,  3.00000000e+00,  4.00000000e+00,  5.00000000e+00,  6.00000000e+00,  7.00000000e+00,  8.00000000e+00,  9.00000000e+00,  1.00000000e+01,  1.10000000e+01, 
  1.20000000e+01,  1.30000000e+01,  1.40000000e+01,  1.50000000e+01])

We initialized a vector of 16 elements above with successive integers from $0$ \(inclusive\) to $16$ \(exclusive\). Next, we use the built-in function [`mindspore.ops.dropout`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/ops/mindspore.ops.dropout.html) to apply dropout to our vector with varying dropout probabilities.

1. $p = 0$: do not apply dropout to any elements. This should return our original vector
1. $p = 0.5$: apply dropout to elements with $50\%$ probability, scaling the rest by a factor of $\frac{1}{1 = 0.5} = 2$
1. $p = 1$: apply dropout to all elements. This should zero out our vector

In [3]:
Y_dropout_none = ops.dropout(X, p=0.0)
Y_dropout_half = ops.dropout(X, p=0.5)
Y_dropout_all = ops.dropout(X, p=1.0)
Y_dropout_none, Y_dropout_half, Y_dropout_all

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: Sy

(Tensor(shape=[16], dtype=Float32, value= [ 0.00000000e+00,  1.00000000e+00,  2.00000000e+00,  3.00000000e+00,  4.00000000e+00,  5.00000000e+00,  6.00000000e+00,  7.00000000e+00,  8.00000000e+00,  9.00000000e+00,  1.00000000e+01,  1.10000000e+01, 
   1.20000000e+01,  1.30000000e+01,  1.40000000e+01,  1.50000000e+01]),
 Tensor(shape=[16], dtype=Float32, value= [ 0.00000000e+00,  0.00000000e+00,  4.00000000e+00,  6.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.20000000e+01,  0.00000000e+00,  1.60000000e+01,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 
   2.40000000e+01,  0.00000000e+00,  0.00000000e+00,  3.00000000e+01]),
 Tensor(shape=[16], dtype=Float32, value= [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 
   0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00]))

## 5.6.3. Concise Implementation

Let's apply the dropout technique to a real, deep neural network. We'll train our deep network on the Fashion MNIST dataset with the following layers.

1. Flatten our images to an unstructured layer with $28 \times 28 \times 1 = 784$ features
1. Apply the first hidden layer with $784$ input channels and $2 ^ 8 = 256$ output channels with ReLU activation
1. Apply $50\%$ dropout to the activations in \(2\) with [`mindspore.nn.Dropout`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/nn/mindspore.nn.Dropout.html)
1. Apply the second hidden layer with $256$ input channels and $256$ output channels with ReLU activation
1. Apply $50\%$ dropout to the activations in \(4\)
1. Apply the final fully connected layer with $256$ input channels and $10$ output channels. Each output channel corresponds to a label in our dataset

Let's load and transform our data. We'll skip the visualization since we've seen the same dataset multiple times already.

In [4]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [5]:
import gzip
import urllib.request

X_train_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-images-idx3-ubyte.gz'
y_train_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-labels-idx1-ubyte.gz'
X_test_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-images-idx3-ubyte.gz'
y_test_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [6]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [7]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(28, 28)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)
train_ds = train_ds.batch(batch_size=512)
test_ds = test_ds.batch(batch_size=10000)

Now define our neural network and train it over $10$ epochs.

In [8]:
import mindspore.nn as nn

net = nn.SequentialCell([
    nn.Flatten(),
    nn.Dense(784, 256, activation='relu'),
    nn.Dropout(p=0.5),
    nn.Dense(256, 256, activation='relu'),
    nn.Dropout(p=0.5),
    nn.Dense(256, 10)
])
net

SequentialCell(
  (0): Flatten()
  (1): Dense(
    input_channels=784, output_channels=256, has_bias=True, activation=ReLU()
    (activation): ReLU()
  )
  (2): Dropout(p=0.5)
  (3): Dense(
    input_channels=256, output_channels=256, has_bias=True, activation=ReLU()
    (activation): ReLU()
  )
  (4): Dropout(p=0.5)
  (5): Dense(input_channels=256, output_channels=10, has_bias=True)
)

In [9]:
import mindspore.amp as amp

amp_level = 'O2'
net_amp = amp.auto_mixed_precision(network=net, amp_level=amp_level)
net_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): Flatten()
    (1): Dense(
      input_channels=784, output_channels=256, has_bias=True, activation=ReLU()
      (activation): ReLU()
    )
    (2): Dropout(p=0.5)
    (3): Dense(
      input_channels=256, output_channels=256, has_bias=True, activation=ReLU()
      (activation): ReLU()
    )
    (4): Dropout(p=0.5)
    (5): Dense(input_channels=256, output_channels=10, has_bias=True)
  )
)

In [10]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [11]:
optimizer = nn.SGD(params=net_amp.trainable_params(), learning_rate=0.1)
optimizer

SGD()

In [12]:
from mindspore.train import Model

model = Model(network=net_amp, loss_fn=loss_fn, optimizer=optimizer)
model

In [13]:
from mindspore.train import LossMonitor

model.train(epoch=10, train_dataset=train_ds, callbacks=[LossMonitor()])

..epoch: 1 step: 1, loss is 2.3166732788085938
epoch: 1 step: 2, loss is 2.29440975189209
epoch: 1 step: 3, loss is 2.2884392738342285
epoch: 1 step: 4, loss is 2.272052764892578
epoch: 1 step: 5, loss is 2.270393133163452
epoch: 1 step: 6, loss is 2.2587599754333496
epoch: 1 step: 7, loss is 2.254704475402832
epoch: 1 step: 8, loss is 2.2371339797973633
epoch: 1 step: 9, loss is 2.232210159301758
epoch: 1 step: 10, loss is 2.2166929244995117
epoch: 1 step: 11, loss is 2.205049991607666
epoch: 1 step: 12, loss is 2.2020363807678223
epoch: 1 step: 13, loss is 2.1793246269226074
epoch: 1 step: 14, loss is 2.172511100769043
epoch: 1 step: 15, loss is 2.147282600402832
epoch: 1 step: 16, loss is 2.1442203521728516
epoch: 1 step: 17, loss is 2.1243598461151123
epoch: 1 step: 18, loss is 2.092128276824951
epoch: 1 step: 19, loss is 2.0779898166656494
epoch: 1 step: 20, loss is 2.044821262359619
epoch: 1 step: 21, loss is 2.024739980697632
epoch: 1 step: 22, loss is 1.9931886196136475
epoch: 

Finally, let's evaluate the validation loss and accuracy of our model. By default, when we call `model.predict`, training mode is set to `False` internally which automatically skips the dropout layers during model inference.

In [14]:
X_test, y_test = next(test_ds.create_tuple_iterator())
X_test.shape, y_test.shape

((10000, 1, 28, 28), (10000, 10))

In [15]:
y_hat = model.predict(X_test)
y_hat.shape

(10000, 10)

In [16]:
validation_loss = loss_fn(y_hat, y_test)
print(f'Validation loss: {validation_loss.asnumpy():.4f}')

Validation loss: 0.4596


In [17]:
y_test = ops.argmax(y_test, dim=1)
y_hat = ops.argmax(y_hat, dim=1)
total = 10000
correct = (y_hat == y_test).sum().item()
validation_accuracy = correct / total
print(f'Validation accuracy: {validation_accuracy:.4f}')

.Validation accuracy: 0.8308


The validation accuracy is somewhere between $80\%$ and $85\%$. It performs slightly better than our simple linear classification model but not as accurate as our first deep model. Perhaps we can obtain better accuracy by fine-tuning the dropout rate after each hidden activation layer and other hyperparameters.

## 5.6.4. Summary

We saw what dropout is, how it works and how to apply it to our deep neural networks as a regularization technique to prevent overfitting.